# ReXKG Pipeline (PyHealth Style)

This notebook shows a PyHealth-native ReXKG workflow similar to other examples:

1. Load dataset
2. Set tasks
3. Build sample datasets
source

predictions_path = Path.cwd() / "result" / "run_relation" / "predictions.json"
if not predictions_path.exists():
    raise FileNotFoundError(f"Missing relation predictions file: {predictions_path}")

structured_output_path = Path.cwd() / "data" / "your_test_file.json"
processed_docs = rexkg_reverse_structure(
    input_json_file=str(predictions_path),
    save_json_file=str(structured_output_path),
)

print("Input predictions:", predictions_path)
print("Structured output:", structured_output_path)
print("Converted documents:", len(processed_docs))

In [ ]:
# Make sure to install the needed libraries used for the rexkg PyHealth files. 
# Kernal for this conda virtual environment is running 3.13.13
#!python -m pip install neraug

In [ ]:
# may take a few mins to run to build cache
from pathlib import Path
import sys
import importlib.util
from pyhealth.metrics import rexkg_reverse_structure

# Make sure the local PyHealth package is importable from this notebook.
# Notebook location: PyHealth/examples/rexkg/load_dataset.ipynb
# Package root:      PyHealth/
pyhealth_root = Path.cwd().resolve().parents[1]
if str(pyhealth_root) not in sys.path:
    sys.path.insert(0, str(pyhealth_root))

from pyhealth.datasets import RexKGDataset #, split_by_patient, get_dataloader
from pyhealth.models import RexKG
from pyhealth.tasks import (
    RexKGEntityExtractionRadiology,
    RexKGRelationExtractionRadiology,
    RexKGKnowledgeGraphConstruction,
)
from pyhealth.metrics import rexkg_reverse_structure

/home/strawhat/miniconda3/envs/cs598-pyhealth/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1) Configure Paths

Set `PROJECT_ROOT` to your repo root if auto-detection does not match your environment.

In [2]:
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "PyHealth").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

REXKG_DATA_ROOT = PROJECT_ROOT / "src" / "ner" / "data"
print("PROJECT_ROOT:", PROJECT_ROOT)
print("REXKG_DATA_ROOT:", REXKG_DATA_ROOT)
print("Data root exists:", REXKG_DATA_ROOT.exists())

PROJECT_ROOT: /mnt/c/Users/Aarje/OneDrive - University of Illinois - Urbana/School/cs598_project/PyHealth/examples
REXKG_DATA_ROOT: /mnt/c/Users/Aarje/OneDrive - University of Illinois - Urbana/School/cs598_project/PyHealth/examples/src/ner/data
Data root exists: False


## 2) Load RexKGDataset - Data Preperation

In [3]:
# data_dir = PROJECT_ROOT / "src" / "ner" / "data"
print("Using PROJECT_ROOT:", PROJECT_ROOT)

expected = PROJECT_ROOT / "df_chexpert_plus_240401.csv"
if not expected.exists():
    raise FileNotFoundError(f"Missing expected CSV: {expected}")

dataset = RexKGDataset(root=str(expected))
dataset.stats()

Using PROJECT_ROOT: /mnt/c/Users/Aarje/OneDrive - University of Illinois - Urbana/School/cs598_project/PyHealth/examples
No config path provided, using default RexKG config
Initializing rexkg dataset from /mnt/c/Users/Aarje/OneDrive - University of Illinois - Urbana/School/cs598_project/PyHealth/examples (dev mode: False)
No cache_dir provided. Using default cache dir: /home/strawhat/.cache/pyhealth/77e4c830-4a0e-5eaf-a252-9cdbff96ceeb
Found cached event dataframe: /home/strawhat/.cache/pyhealth/77e4c830-4a0e-5eaf-a252-9cdbff96ceeb/global_event_df.parquet
Dataset: rexkg
Dev mode: False
Number of patients: 27361
Number of events: 59441


## 3) Apply ReXKG Tasks - Run Entity Pipeline

In [4]:
entity_task = RexKGEntityExtractionRadiology()
relation_task = RexKGRelationExtractionRadiology()
kg_task = RexKGKnowledgeGraphConstruction()

# Prefer repo-relative split files created by src/ner/data/structure_data.py
split_root = PROJECT_ROOT / "src" / "ner" / "data" / "data_split"
if not split_root.exists():
    # Fallback when running from PyHealth/examples/rexkg
    split_root = Path.cwd() / "data" / "data_split"
train_json = split_root / "train.json"
dev_json = split_root / "test.json"
test_json = split_root / "test.json"

for p in [train_json, dev_json, test_json]:
    if not p.exists():
        raise FileNotFoundError(f"Missing split file: {p}")

# Force output under this notebook folder.
entity_output_dir = Path.cwd() / "result" / "run_entity"
metrics = RexKGEntityExtractionRadiology.run_entity_pipeline(
    train_data=str(train_json),
    dev_data=str(dev_json),
    test_data=str(test_json),
    model="bert-base-uncased",
    output_dir=str(entity_output_dir),
    do_train=True,
    do_eval=True,
    eval_test=True,
    learning_rate=1e-5,
    task_learning_rate=5e-4,
    train_batch_size=8,
    eval_batch_size=64,
    num_epoch=1,
    context_window=100,
)
print(metrics)

pred_file = entity_output_dir / "ent_pred_mimic_headct.json"
print("Expected prediction file:", pred_file)
print("Prediction file exists:", pred_file.exists())

entity_samples = dataset.set_task(entity_task)





# relation_samples = dataset.set_task(relation_task)
# kg_samples = dataset.set_task(kg_task)

# print("Entity samples:", len(entity_samples))
# print("Relation samples:", len(relation_samples))
# print("KG samples:", len(kg_samples))

Some weights of BertForEntity were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['ner_classifier.0.network.0.bias', 'ner_classifier.0.network.0.weight', 'ner_classifier.0.network.3.bias', 'ner_classifier.0.network.3.weight', 'ner_classifier.1.bias', 'ner_classifier.1.weight', 'width_embedding.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
  0%|          | 0/1 [00:00<?, ?it/s]

Evaluating...
Accuracy: 0.973876
Cor: 2720, Pred TOT: 3631, Gold TOT: 3699
P: 0.74910, R: 0.73533, F1: 0.74216
Used time: 3.409109
Saving model to /mnt/c/Users/Aarje/OneDrive - University of Illinois - Urbana/School/cs598_project/PyHealth/examples/rexkg/result/run_entity...


100%|██████████| 1/1 [00:48<00:00, 48.27s/it]


Evaluating...
Accuracy: 0.973876
Cor: 2720, Pred TOT: 3631, Gold TOT: 3699
P: 0.74910, R: 0.73533, F1: 0.74216
Used time: 3.739961
Total pred entities: 3631
Output predictions to /mnt/c/Users/Aarje/OneDrive - University of Illinois - Urbana/School/cs598_project/PyHealth/examples/rexkg/result/run_entity/ent_pred_mimic_headct.json..
{'output_dir': '/mnt/c/Users/Aarje/OneDrive - University of Illinois - Urbana/School/cs598_project/PyHealth/examples/rexkg/result/run_entity', 'log_file': '/mnt/c/Users/Aarje/OneDrive - University of Illinois - Urbana/School/cs598_project/PyHealth/examples/rexkg/result/run_entity/train.log', 'task': 'mimic01', 'model': 'bert-base-uncased', 'best_dev_f1': 0.7421555252387448, 'test_f1': 0.7421555252387448, 'test_prediction_file': '/mnt/c/Users/Aarje/OneDrive - University of Illinois - Urbana/School/cs598_project/PyHealth/examples/rexkg/result/run_entity/ent_pred_mimic_headct.json'}
Expected prediction file: /mnt/c/Users/Aarje/OneDrive - University of Illinois -

## 3.5) Run Relation Pipeline via PyHealth Import

All input/output paths below stay inside `PyHealth/examples/rexkg/`.

In [ ]:
import importlib
import pyhealth.tasks.rexkg as rexkg_module

importlib.reload(rexkg_module)
RexKGRelationExtractionRadiology = rexkg_module.RexKGRelationExtractionRadiology

relation_output_dir = Path.cwd() / "result" / "run_relation"
relation_metrics = RexKGRelationExtractionRadiology.run_relation_pipeline(
    train_file=str(train_json),
    entity_output_dir=str(entity_output_dir),
    entity_predictions_dev="ent_pred_mimic_headct.json",
    entity_predictions_test="ent_pred_mimic_headct.json",
    model="bert-base-uncased",
    output_dir=str(relation_output_dir),
    do_train=True,
    do_eval=True,
    eval_with_gold=True,
    do_lower_case=True,
    train_batch_size=16,
    eval_batch_size=32,
    learning_rate=5e-5,
    num_train_epochs=1,
    # context_window=100,
    context_window=20,
    max_seq_length=256,
)
print(relation_metrics)

relation_pred_file = relation_output_dir / "predictions.json"
print("Expected relation prediction file:", relation_pred_file)
print("Relation prediction file exists:", relation_pred_file.exists())

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


{'output_dir': '/mnt/c/Users/Aarje/OneDrive - University of Illinois - Urbana/School/cs598_project/PyHealth/examples/rexkg/result/run_relation', 'prediction_file': '/mnt/c/Users/Aarje/OneDrive - University of Illinois - Urbana/School/cs598_project/PyHealth/examples/rexkg/result/run_relation/predictions.json', 'log_file': '/mnt/c/Users/Aarje/OneDrive - University of Illinois - Urbana/School/cs598_project/PyHealth/examples/rexkg/result/run_relation/train.log', 'task': 'mimic01', 'model': 'bert-base-uncased', 'accuracy': 0.9496258096975033, 'precision': 0.7098056537102474, 'recall': 0.6061863447755564, 'f1': 0.653916581892167}
Expected relation prediction file: /mnt/c/Users/Aarje/OneDrive - University of Illinois - Urbana/School/cs598_project/PyHealth/examples/rexkg/result/run_relation/predictions.json
Relation prediction file exists: True


: 

: 

# Step 5 Inference and Evaluation

In [5]:


predictions_path = Path.cwd() / "result" / "run_relation" / "predictions.json"
if not predictions_path.exists():
    raise FileNotFoundError(f"Missing relation predictions file: {predictions_path}")

structured_output_path = Path.cwd() / "data" / "your_test_file.json"
processed_docs = rexkg_reverse_structure(
    input_json_file=str(predictions_path),
    save_json_file=str(structured_output_path),
)

print("Input predictions:", predictions_path)
print("Structured output:", structured_output_path)
print("Converted documents:", len(processed_docs))

Input predictions: /mnt/c/Users/Aarje/OneDrive - University of Illinois - Urbana/School/cs598_project/PyHealth/examples/rexkg/result/run_relation/predictions.json
Structured output: /mnt/c/Users/Aarje/OneDrive - University of Illinois - Urbana/School/cs598_project/PyHealth/examples/rexkg/data/your_test_file.json
Converted documents: 560
